# Kickstarter — Linguistic Patterns & Campaign Success

**Obiettivo:** stimare l'associazione tra token/bigrammi nel testo delle descrizioni e il successo della campagna, controllando per variabili strutturali.

### Struttura
1. Setup & caricamento dati
2. Feature engineering
3. Frequency analysis (raw difference + log-odds)
4. Modello globale — unigram
5. Modello intra-categoria — unigram
6. Modello globale — bigram
7. Modello intra-categoria — bigram

---

**Nota metodologica — modello:** la regressione logistica è uno strumento interpretativo, non predittivo. I coefficienti stimano l'associazione condizionata tra la presenza di un token e la probabilità di successo, dati i controlli. Non si fa causal inference.

**Nota metodologica — valutazione:** l'unica metrica riportata è l'AUC-ROC in 5-fold cross-validation stratificata. Non è usata in senso predittivo: serve come sanity check sul modello. Un AUC vicino a 0.5 indicherebbe che il modello non discrimina tra successo e fallimento meglio del caso, rendendo i coefficienti estratti poco affidabili. Un AUC sostanzialmente superiore a 0.5 conferma che il modello cattura segnale reale nei dati.

**Nota metodologica — robustezza dei coefficienti:** i token vengono filtrati per `doc_freq >= MIN_DF`. Coefficienti su termini molto rari sono instabili anche se di grande magnitudine: il filtro riduce questo rischio.

## 1. Setup & caricamento dati

In [3]:
import ast
import warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from scipy.sparse import hstack

warnings.filterwarnings('ignore')

# ── Parametri globali ──────────────────────────────────────────────────────────
MIN_DF_UNIGRAM = 20      # soglia minima doc frequency — unigram
MIN_DF_BIGRAM  = 20      # soglia minima doc frequency — bigram
BASELINE_CAT   = 'Film & Video'
CV_FOLDS       = 5

In [4]:
df = pd.read_csv('Kickstarter_processed (1).csv')
df['description_processed'] = df['description_processed'].apply(ast.literal_eval)

print(f'Shape: {df.shape}')
print(f'\nCategorie:\n{df["category"].value_counts()}')
print(f'\nStatus (1=success, 0=failure):\n{df["status"].value_counts()}')
df.head(3)

Shape: (7354, 16)

Categorie:
category
Film & Video    2011
Technology      1489
Games           1422
Music           1218
Publishing      1214
Name: count, dtype: int64

Status (1=success, 0=failure):
status
1    4510
0    2844
Name: count, dtype: int64


,Unnamed: 0,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency,category,video,reached,status,duration,description_processed,pos_tagged
0,0,https://www.kickstarter.com/projects/thetruthb...,NaN,The PROBLEM: So much entertainment today pushe...,71123.0,71123.0,61607,48000.0,USD,Film & Video,1,148.172917,1,30,"[entertainment, today, push, unholy, agenda, c...","[('problem', 'NN'), ('much', 'JJ'), ('entertai..."
1,1,https://www.kickstarter.com/projects/99625582/...,NaN,Millions of American college students have stu...,65318.0,65318.0,56579,61500.0,USD,Film & Video,1,106.208130,1,41,"[million, american, college, student, study, a...","[('millions', 'NNS'), ('american', 'JJ'), ('co..."
2,2,https://www.kickstarter.com/projects/distortre...,Cartoon Network Alphabet Pins,Full A-Z Set I'm launching this set to show my...,462.0,462.0,400,8000.0,USD,Film & Video,1,5.775000,0,45,"[cartoon, network, type, letter, instead, lett...","[('full', 'JJ'), ('set', 'NN'), ('launching', ..."


## 2. Feature engineering

In [5]:
df['goal_log']    = np.log(df['goal'])
df['text_length'] = df['description_processed'].apply(len)
df['text_joined'] = df['description_processed'].apply(lambda x: ' '.join(x))

# Dummy di categoria — baseline: Film & Video
cat_dummies = pd.get_dummies(df['category'], drop_first=False, dtype=float)
cat_dummies = cat_dummies.drop(columns=[BASELINE_CAT])

CONTROLS   = ['goal_log', 'duration', 'text_length', 'video']
CATEGORIES = df['category'].unique().tolist()

print('Controlli numerici:', CONTROLS)
print('Dummy categoria (baseline =', BASELINE_CAT + '):', list(cat_dummies.columns))
df[CONTROLS + ['status']].describe().round(3)

Controlli numerici: ['goal_log', 'duration', 'text_length', 'video']
Dummy categoria (baseline = Film & Video): ['Games', 'Music', 'Publishing', 'Technology']


,goal_log,duration,text_length,video,status
count,7354.000,7354.000,7354.000,7354.000,7354.000
mean,9.680,34.443,291.292,0.877,0.613
std,0.805,11.260,245.452,0.328,0.487
min,8.517,3.000,15.000,0.000,0.000
25%,9.148,30.000,120.000,1.000,0.000
50%,9.473,30.000,220.000,1.000,1.000
75%,10.127,37.000,386.000,1.000,1.000
max,13.816,91.000,1965.000,1.000,1.000


## 3. Frequency analysis — descrittiva

Per ogni token calcoliamo:
- **Raw difference**: frequenza relativa nei successi − frequenza relativa nei fallimenti
- **Log-odds ratio**: log[P(token|success)/(1−P)] − log[P(token|failure)/(1−P)], con smoothing di Laplace. Più robusto della raw difference perché penalizza i token rari e normalizza per la base rate di ciascun gruppo.

In [6]:
def frequency_analysis(df_sub, min_doc_freq=MIN_DF_UNIGRAM):
    df_s     = df_sub[df_sub['status'] == 1]
    df_f     = df_sub[df_sub['status'] == 0]
    tokens_s = [t for doc in df_s['description_processed'] for t in doc]
    tokens_f = [t for doc in df_f['description_processed'] for t in doc]
    freq_s   = Counter(tokens_s)
    freq_f   = Counter(tokens_f)
    doc_freq = Counter(t for doc in df_sub['description_processed'] for t in set(doc))
    vocab    = {w for w, c in doc_freq.items() if c >= min_doc_freq}
    total_s  = len(tokens_s)
    total_f  = len(tokens_f)
    rows = []
    for w in vocab:
        p_s      = (freq_s.get(w, 0) + 1) / (total_s + len(vocab))  # Laplace smoothing
        p_f      = (freq_f.get(w, 0) + 1) / (total_f + len(vocab))
        raw_diff = (freq_s.get(w, 0) / total_s) - (freq_f.get(w, 0) / total_f)
        log_odds = np.log(p_s / (1 - p_s)) - np.log(p_f / (1 - p_f))
        rows.append({
            'token':    w,
            'doc_freq': doc_freq[w],
            'raw_diff': round(raw_diff, 6),
            'log_odds': round(log_odds, 4)
        })
    return pd.DataFrame(rows).sort_values('log_odds', ascending=False).reset_index(drop=True)


# — globale —
freq_global = frequency_analysis(df)
print('=== TOP 10 associati al SUCCESSO (log-odds, globale) ===')
display(freq_global.head(10))
print('\n=== TOP 10 associati al FALLIMENTO (log-odds, globale) ===')
display(freq_global.tail(10).sort_values('log_odds'))

=== TOP 10 associati al SUCCESSO (log-odds, globale) ===


,token,doc_freq,raw_diff,log_odds
0,cine,24,0.000106,3.6786
1,compendium,39,0.000165,3.4357
2,drivethrurpg,32,0.000076,3.3576
3,celeste,20,0.000030,3.1024
4,avalanche,23,0.000030,3.1024
5,harlem,21,0.000028,3.0548
6,meow,21,0.000131,2.9997
7,gst,39,0.000045,2.8528
8,newborn,25,0.000023,2.8377
9,devin,24,0.000022,2.8069



=== TOP 10 associati al FALLIMENTO (log-odds, globale) ===


,token,doc_freq,raw_diff,log_odds
9305,ableton,20,-0.000161,-2.9143
9304,spacecraft,28,-0.000114,-2.7497
9303,wearer,24,-0.000035,-2.6048
9302,stud,20,-0.000049,-2.5307
9301,balloon,27,-0.000099,-2.2808
9300,savannah,34,-0.000198,-2.2747
9299,herb,35,-0.000141,-2.2684
9298,facilitator,27,-0.000047,-2.2430
9297,golfer,27,-0.000138,-2.2083
9296,altitude,21,-0.000038,-2.1787


In [7]:
# — intra-categoria —
for cat in CATEGORIES:
    freq_cat = frequency_analysis(df[df['category'] == cat])
    print(f'\n=== {cat} — TOP 10 SUCCESSO ===')
    display(freq_cat.head(10))
    print(f'=== {cat} — TOP 10 FALLIMENTO ===')
    display(freq_cat.tail(10).sort_values('log_odds'))


=== Film & Video — TOP 10 SUCCESSO ===


,token,doc_freq,raw_diff,log_odds
0,cine,22,0.000390,4.2115
1,addon,37,0.000314,2.9359
2,activism,30,0.000101,2.8776
3,christina,21,0.000093,2.7975
4,oakland,24,0.000156,2.6481
5,alice,27,0.000287,2.5815
6,beth,21,0.000140,2.5463
7,emmynominated,24,0.000069,2.5098
8,disc,27,0.000124,2.4329
9,afi,22,0.000061,2.3920


=== Film & Video — TOP 10 FALLIMENTO ===


,token,doc_freq,raw_diff,log_odds
3960,sheriff,22,-0.000165,-1.9802
3959,ninja,20,-0.000219,-1.9314
3958,martinez,25,-0.000174,-1.8048
3957,gate,34,-0.000460,-1.8035
3956,flash,32,-0.000238,-1.7878
3955,bend,26,-0.000241,-1.7539
3954,hair,106,-0.000996,-1.7205
3953,hood,23,-0.000178,-1.6513
3952,riproduci,44,-0.000699,-1.6251
3951,suono,44,-0.000466,-1.6205



=== Games — TOP 10 SUCCESSO ===


...

,token,doc_freq,raw_diff,log_odds
2681,lifestyle,42,-0.000408,-2.1143
2680,murder,34,-0.000578,-2.0403
2679,marriage,26,-0.000345,-2.0401
2678,seasonal,21,-0.000279,-2.0147
2677,intellectual,21,-0.000125,-1.9345
2676,revenue,22,-0.000175,-1.9266
2675,district,20,-0.000274,-1.9194
2674,ad,36,-0.000336,-1.8859
2673,tablet,25,-0.000187,-1.8700
2672,trailer,28,-0.000249,-1.8393



=== Technology — TOP 10 SUCCESSO ===


,token,doc_freq,raw_diff,log_odds
0,carrier,20,0.000554,3.2424
1,jellop,80,0.000292,3.1170
2,nozzle,22,0.000205,2.7829
3,backercrew,25,0.000086,2.3467
4,highprecision,20,0.000075,2.2189
5,vat,28,0.000159,2.0828
6,sdk,21,0.000184,2.0547
7,addons,87,0.000397,2.0521
8,unbox,23,0.000089,2.0182
9,splash,21,0.000195,1.9695


=== Technology — TOP 10 FALLIMENTO ===


,token,doc_freq,raw_diff,log_odds
3068,faith,21,-0.000204,-2.9519
3067,chord,23,-0.000850,-2.7156
3066,trail,26,-0.000501,-2.3527
3064,empowerment,24,-0.000161,-2.2587
3065,mvp,23,-0.000161,-2.2587
3063,satellite,27,-0.000384,-2.1151
3062,echo,23,-0.000218,-2.0075
3061,income,25,-0.000116,-1.9710
3060,airplane,33,-0.000507,-1.9638
3059,theory,38,-0.000204,-1.9524


## Funzioni di supporto per i modelli

In [8]:
def make_model():
    """Regressione logistica L2, class_weight bilanciato."""
    return LogisticRegression(
        penalty='l2',
        solver='lbfgs',
        max_iter=500,
        class_weight='balanced'
    )


def evaluate_auc(X, y):
    """
    AUC-ROC in 5-fold CV stratificata.
    Usata come sanity check: verifica che il modello discrimini
    tra successo e fallimento meglio del caso (AUC >> 0.5).
    Non è una valutazione predittiva.
    """
    cv     = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    scores = cross_val_score(make_model(), X, y, cv=cv, scoring='roc_auc')
    return pd.DataFrame({
        'AUC-ROC mean': [round(scores.mean(), 3)],
        'AUC-ROC std':  [round(scores.std(),  3)]
    })


def extract_coefs(model, feature_names, X_text, min_doc_freq, col_name='word'):
    """Estrae coefficienti sui token con filtro doc_freq."""
    coef     = model.coef_[0][:len(feature_names)]
    doc_freq = np.asarray((X_text > 0).sum(axis=0)).flatten()
    info = pd.DataFrame({
        col_name:   feature_names,
        'coef':     coef.round(4),
        'doc_freq': doc_freq
    })
    return info[info['doc_freq'] >= min_doc_freq].copy().reset_index(drop=True)


def show_top(info, col_name, n=10):
    """Stampa top n token per successo e fallimento."""
    cols = [col_name, 'coef', 'doc_freq']
    print(f'\n--- TOP {n} associati al SUCCESSO ---')
    display(info.sort_values('coef', ascending=False).head(n)[cols].reset_index(drop=True))
    print(f'\n--- TOP {n} associati al FALLIMENTO ---')
    display(info.sort_values('coef').head(n)[cols].reset_index(drop=True))

## 4. Modello globale — unigram

**Spec:** `P(success) ~ unigram BoW + goal_log + duration + text_length + video + categoria_dummy`  
Baseline categoria: Film & Video.

In [9]:
y = df['status'].values

vec_uni_glob    = CountVectorizer(min_df=MIN_DF_UNIGRAM)
X_text_uni_glob = vec_uni_glob.fit_transform(df['text_joined'])
feat_uni_glob   = vec_uni_glob.get_feature_names_out()

scaler_glob = StandardScaler()
X_num_glob  = sp.csr_matrix(scaler_glob.fit_transform(df[CONTROLS]))
X_cat_glob  = sp.csr_matrix(cat_dummies.values)

X_global_uni = hstack([X_text_uni_glob, X_num_glob, X_cat_glob])

# fit
model_global_uni = make_model()
model_global_uni.fit(X_global_uni, y)

# AUC-ROC (sanity check)
print('=== AUC-ROC — modello globale unigram (5-fold CV) ===')
display(evaluate_auc(X_global_uni, y))

=== AUC-ROC — modello globale unigram (5-fold CV) ===


,AUC-ROC mean,AUC-ROC std
0,0.807,0.014


In [10]:
info_uni_glob = extract_coefs(
    model_global_uni, feat_uni_glob, X_text_uni_glob,
    min_doc_freq=MIN_DF_UNIGRAM, col_name='word'
)

print('=== COEFFICIENTI TOKEN — modello globale unigram ===')
show_top(info_uni_glob, col_name='word')

=== COEFFICIENTI TOKEN — modello globale unigram ===

--- TOP 10 associati al SUCCESSO ---


,word,coef,doc_freq
0,dramatic,1.1369,255
1,jellop,1.0468,138
2,tariff,0.9384,153
3,publicist,0.9343,70
4,netflix,0.9006,209
5,transfer,0.8723,197
6,decides,0.8569,68
7,writerdirector,0.8445,180
8,disappear,0.8356,147
9,seriously,0.8347,186



--- TOP 10 associati al FALLIMENTO ---


,word,coef,doc_freq
0,maria,-1.1234,83
1,laughter,-0.9383,121
2,gypsy,-0.9155,22
3,non,-0.8923,69
4,visuals,-0.8875,282
5,rhyme,-0.8166,61
6,cbs,-0.8102,56
7,behindthescenes,-0.7999,279
8,therefore,-0.7806,206
9,kill,-0.7770,343


## 5. Modello intra-categoria — unigram

**Spec per categoria:** `P(success) ~ unigram BoW + goal_log + duration + text_length + video`  
Nessuna dummy di categoria (modello già su subset).

In [11]:
def run_intracat(df, cat, ngram_range=(1,1), min_df=MIN_DF_UNIGRAM, col_name='word'):
    """Fit logistica su subset categoria + AUC-ROC + estrazione coef."""
    df_cat = df[df['category'] == cat].copy().reset_index(drop=True)
    y_cat  = df_cat['status'].values

    vec    = CountVectorizer(ngram_range=ngram_range, min_df=min_df)
    X_text = vec.fit_transform(df_cat['text_joined'])
    feat   = vec.get_feature_names_out()

    scaler = StandardScaler()
    X_num  = sp.csr_matrix(scaler.fit_transform(df_cat[CONTROLS]))
    X      = hstack([X_text, X_num])

    model  = make_model()
    model.fit(X, y_cat)

    auc_df = evaluate_auc(X, y_cat)
    info   = extract_coefs(model, feat, X_text, min_doc_freq=min_df, col_name=col_name)

    return info, auc_df


for cat in CATEGORIES:
    print(f'\n===============================')
    print(f'  {cat} — unigram')
    print(f'===============================')
    info, auc_df = run_intracat(df, cat, ngram_range=(1,1))
    print('--- AUC-ROC (5-fold CV) ---')
    display(auc_df)
    show_top(info, col_name='word')


  Film & Video — unigram
--- AUC-ROC (5-fold CV) ---


,AUC-ROC mean,AUC-ROC std
0,0.816,0.011



--- TOP 10 associati al SUCCESSO ---


,word,coef,doc_freq
0,netflix,0.7140,172
1,hometown,0.7021,79
2,thesis,0.6973,59
3,dollar,0.6630,333
4,jeremy,0.6470,33
5,email,0.6320,173
6,wake,0.6275,70
7,expand,0.6184,200
8,documentary,0.6167,516
9,chris,0.6118,139



--- TOP 10 associati al FALLIMENTO ---


...

,AUC-ROC mean,AUC-ROC std
0,0.815,0.013



--- TOP 10 associati al SUCCESSO ---


,word,coef,doc_freq
0,promote,0.7303,173
1,recognize,0.6958,122
2,printing,0.5757,64
3,mini,0.5641,89
4,review,0.5253,194
5,anywhere,0.5077,254
6,specification,0.5076,184
7,news,0.4781,119
8,less,0.4695,317
9,quickly,0.4690,268



--- TOP 10 associati al FALLIMENTO ---


,word,coef,doc_freq
0,hold,-0.5165,287
1,expand,-0.5114,249
2,site,-0.4999,143
3,patent,-0.4832,222
4,person,-0.4702,179
5,similar,-0.4640,174
6,remove,-0.4403,180
7,fashion,-0.4348,53
8,energy,-0.4311,186
9,cloud,-0.4275,118


## 6. Modello globale — bigram

**Spec:** identico al globale unigram con `ngram_range=(2,2)` e `min_df=20`.

In [12]:
vec_bi_glob    = CountVectorizer(ngram_range=(2,2), min_df=MIN_DF_BIGRAM)
X_text_bi_glob = vec_bi_glob.fit_transform(df['text_joined'])
feat_bi_glob   = vec_bi_glob.get_feature_names_out()

X_global_bi = hstack([X_text_bi_glob, X_num_glob, X_cat_glob])

model_global_bi = make_model()
model_global_bi.fit(X_global_bi, y)

print('=== AUC-ROC — modello globale bigram (5-fold CV) ===')
display(evaluate_auc(X_global_bi, y))

=== AUC-ROC — modello globale bigram (5-fold CV) ===


,AUC-ROC mean,AUC-ROC std
0,0.776,0.016


In [13]:
info_bi_glob = extract_coefs(
    model_global_bi, feat_bi_glob, X_text_bi_glob,
    min_doc_freq=MIN_DF_BIGRAM, col_name='bigram'
)

print('=== COEFFICIENTI BIGRAM — modello globale ===')
show_top(info_bi_glob, col_name='bigram')

=== COEFFICIENTI BIGRAM — modello globale ===

--- TOP 10 associati al SUCCESSO ---


,bigram,coef,doc_freq
0,premier sundance,2.0768,37
1,fully compatible,1.9400,40
2,fully manage,1.8633,22
3,proud introduce,1.8533,23
4,promote jellop,1.8202,137
5,sell separately,1.7955,25
6,fair price,1.7139,21
7,deluxe hardcover,1.6799,29
8,source inspiration,1.6266,23
9,language culture,1.5605,20



--- TOP 10 associati al FALLIMENTO ---


,bigram,coef,doc_freq
0,gain exclusive,-2.0758,22
1,delve deep,-1.8069,22
2,fairly compensate,-1.6422,25
3,must fight,-1.5071,20
4,law enforcement,-1.4918,23
5,whose passion,-1.4881,24
6,social post,-1.4113,20
7,whats schedule,-1.3519,22
8,per episode,-1.3387,20
9,download link,-1.3231,25


## 7. Modello intra-categoria — bigram

In [14]:
for cat in CATEGORIES:
    print(f'\n===============================')
    print(f'  {cat} — bigram')
    print(f'===============================')
    info, auc_df = run_intracat(
        df, cat, ngram_range=(2,2), min_df=MIN_DF_BIGRAM, col_name='bigram'
    )
    print('--- AUC-ROC (5-fold CV) ---')
    display(auc_df)
    show_top(info, col_name='bigram')


  Film & Video — bigram
--- AUC-ROC (5-fold CV) ---


,AUC-ROC mean,AUC-ROC std
0,0.731,0.018



--- TOP 10 associati al SUCCESSO ---


,bigram,coef,doc_freq
0,premier sundance,2.1391,36
1,across america,1.5593,21
2,costume designer,1.4279,45
3,critically acclaim,1.3936,41
4,million view,1.3208,48
5,credit imdb,1.2760,27
6,york los,1.2252,27
7,comedy central,1.2159,33
8,archival footage,1.2127,41
9,murder mystery,1.1610,20



--- TOP 10 associati al FALLIMENTO ---


...

,AUC-ROC mean,AUC-ROC std
0,0.753,0.019



--- TOP 10 associati al SUCCESSO ---


,bigram,coef,doc_freq
0,fully compatible,1.8306,21
1,fully manage,1.7214,22
2,send survey,1.4484,20
3,promote jellop,1.3136,79
4,across country,1.2665,20
5,press kit,1.0983,25
6,field view,1.0249,23
7,android phone,0.9532,27
8,current state,0.9222,20
9,via bluetooth,0.9166,47



--- TOP 10 associati al FALLIMENTO ---


,bigram,coef,doc_freq
0,price point,-1.3066,30
1,cell phone,-0.9218,34
2,front view,-0.8871,20
3,hour continuous,-0.8795,20
4,usb cable,-0.7758,55
5,smart phone,-0.7719,47
6,solder iron,-0.6108,22
7,input output,-0.5910,49
8,answer question,-0.5565,30
9,bluetooth connection,-0.5527,31


#### The analysis shows that the textual descriptions of Kickstarter projects contain meaningful information related to project outcomes. In particular, the unigram-based model achieves AUC values around 0.8, indicating a non-trivial ability to discriminate between successful and failed projects. This result remains stable in the intra-category analysis, suggesting that the informational content of the text is not solely driven by differences across categories, but also holds within them.
#### From a qualitative perspective, a consistent pattern emerges across specifications. Terms and expressions associated with success tend to be more specific, concrete, and product-oriented, often referring to verifiable elements such as technical features, production details, or distribution. In contrast, terms associated with failure are more generic, descriptive, or promotional, and less tied to concrete aspects of the project. This pattern is further reinforced by the bigram analysis, which highlights that not only individual words, but also the way they are combined, reflects different communication styles.
#### Overall, the results suggest that the language used in project descriptions is not neutral, but reflects underlying communication strategies and project characteristics that are systematically associated with campaign outcomes. However, since the analysis is associative, these findings should not be interpreted causally: the identified terms do not determine success, but rather serve as indicators of contexts, content, and expressive styles that are more frequently observed in successful projects compared to unsuccessful ones.

### Generative Task

In [15]:
# ============================================================
# TASK GENERATIVA — setup intra-categoria riutilizzabile
# ============================================================

generative_models = {}

for cat in CATEGORIES:
    df_cat = df[df['category'] == cat].copy().reset_index(drop=True)
    y_cat  = df_cat['status'].values

    # testo: bigrammi intra-categoria
    vec = CountVectorizer(ngram_range=(2, 2), min_df=MIN_DF_BIGRAM)
    X_text = vec.fit_transform(df_cat['text_joined'])
    feat = vec.get_feature_names_out()

    # controlli numerici
    scaler = StandardScaler()
    X_num = sp.csr_matrix(scaler.fit_transform(df_cat[CONTROLS]))

    # matrice finale
    X = hstack([X_text, X_num])

    # modello
    model = make_model()
    model.fit(X, y_cat)

    # tabella coefficienti
    info = extract_coefs(
        model=model,
        feature_names=feat,
        X_text=X_text,
        min_doc_freq=MIN_DF_BIGRAM,
        col_name='bigram'
    )

    generative_models[cat] = {
        'df_cat': df_cat,
        'y_cat': y_cat,
        'vec': vec,
        'feat': feat,
        'scaler': scaler,
        'X_text': X_text,
        'X_num': X_num,
        'X': X,
        'model': model,
        'info': info.sort_values('coef', ascending=False).reset_index(drop=True)
    }

print(f'Modelli generativi creati per {len(generative_models)} categorie.')
print('Categorie disponibili:')
print(list(generative_models.keys()))

Modelli generativi creati per 5 categorie.
Categorie disponibili:
['Film & Video', 'Games', 'Music', 'Publishing', 'Technology']


In [16]:
# CELLA 2 — fit intra-categoria bigram e salvataggio oggetti utili

gen_models_bi = {}

for cat in CATEGORIES:
    df_cat = df[df['category'] == cat].copy().reset_index(drop=True)
    y_cat  = df_cat['status'].values

    vec    = CountVectorizer(ngram_range=(2, 2), min_df=MIN_DF_BIGRAM)
    X_text = vec.fit_transform(df_cat['text_joined'])
    feat   = vec.get_feature_names_out()

    scaler = StandardScaler()
    X_num  = sp.csr_matrix(scaler.fit_transform(df_cat[CONTROLS]))
    X_cat  = hstack([X_text, X_num])

    model  = make_model()
    model.fit(X_cat, y_cat)

    info = extract_coefs(
        model=model,
        feature_names=feat,
        X_text=X_text,
        min_doc_freq=MIN_DF_BIGRAM,
        col_name='bigram'
    ).sort_values('coef', ascending=False).reset_index(drop=True)

    gen_models_bi[cat] = {
        'df_cat':  df_cat,
        'y_cat':   y_cat,
        'vec':     vec,
        'X_text':  X_text,
        'feat':    feat,
        'scaler':  scaler,
        'X_num':   X_num,
        'X':       X_cat,
        'model':   model,
        'info':    info
    }

print(f'Categorie salvate: {len(gen_models_bi)}')
print(list(gen_models_bi.keys()))

Categorie salvate: 5
['Film & Video', 'Games', 'Music', 'Publishing', 'Technology']


In [17]:
# CELLA 3 — scelta categoria + progetto fallito casuale + score iniziale

CAT = 'Music'          # cambia categoria se vuoi
RANDOM_STATE = 42

res = gen_models_bi[CAT]
df_cat = res['df_cat']

failure_sample = (
    df_cat[df_cat['status'] == 0]
    .sample(1, random_state=RANDOM_STATE)
    .copy()
)

row = failure_sample.iloc[0]

X_text_old = res['vec'].transform([row['text_joined']])
X_num_old  = sp.csr_matrix(res['scaler'].transform(failure_sample[CONTROLS]))
X_old      = hstack([X_text_old, X_num_old])

p_old     = res['model'].predict_proba(X_old)[0, 1]
logit_old = res['model'].decision_function(X_old)[0]

print('Categoria:', CAT)
print('Probabilità iniziale di successo:', round(p_old, 4))
print('Score lineare iniziale:', round(logit_old, 4))
print('\nDescrizione originale preprocessata:\n')
print(row['text_joined'][:2000])

Categoria: Music
Probabilità iniziale di successo: 0.3353
Score lineare iniziale: -0.6845

Descrizione originale preprocessata:

photo henry hung overview portrait ancestor solo singersongwriter andrew across weave grief memory ancestral echo intimate confession layered harmony serve sonic document generation discover study interpret overview andrew hehim capture felt human century visual storytelling depth resilience human spirit weave race gender identity ongoing archive human seek understand remember self reconcile personal struggle universal truth uncover beauty meaning chaos existence objective distribution portrait ancestor fair compensation collaborator aim invite listener shape body root memory legacy cultural storytelling theme andrew intersection identity legacy resilience examine personal cultural history shape carry forward


In [19]:
# CELLA 4 — versione con filtro aggiuntivo sulla frequenza documentale

MIN_DOC_FREQ_GEN = 30

top_success_bigrams = (
    res['info'][res['info']['doc_freq'] >= MIN_DOC_FREQ_GEN]
    .sort_values(['coef', 'doc_freq'], ascending=[False, False])
    .reset_index(drop=True)
)

display(top_success_bigrams.head(20))

,bigram,coef,doc_freq
0,radio station,1.8294,31
1,united state,0.4019,37
2,across country,0.3693,46
3,capable replay,0.2466,39
4,los angeles,0.0970,56
5,rock roll,0.0307,34
6,grammy win,0.0232,36
7,physical cd,-0.1117,33
8,hip hop,-0.1446,34
9,singer songwriter,-0.4044,30


In [20]:
# CELLA 5 — riscrittura manuale + nuova probabilità stimata

original_text = row['text_joined']

rewritten_text = original_text + " high quality original music professional recording live performance award winning team"

X_text_new = res['vec'].transform([rewritten_text])
X_num_new  = sp.csr_matrix(res['scaler'].transform(failure_sample[CONTROLS]))
X_new      = hstack([X_text_new, X_num_new])

p_new     = res['model'].predict_proba(X_new)[0, 1]
logit_new = res['model'].decision_function(X_new)[0]

comparison = pd.DataFrame({
    'version': ['original', 'rewritten'],
    'prob_success': [p_old, p_new],
    'logit_score': [logit_old, logit_new]
})

comparison['delta_prob']  = comparison['prob_success'] - comparison.loc[0, 'prob_success']
comparison['delta_logit'] = comparison['logit_score'] - comparison.loc[0, 'logit_score']

display(comparison)

print('\nTESTO RISCRITTO:\n')
print(rewritten_text[:2500])

,version,prob_success,logit_score,delta_prob,delta_logit
0,original,0.335261,-0.684487,0.0,0.0
1,rewritten,0.335261,-0.684487,0.0,0.0



TESTO RISCRITTO:

photo henry hung overview portrait ancestor solo singersongwriter andrew across weave grief memory ancestral echo intimate confession layered harmony serve sonic document generation discover study interpret overview andrew hehim capture felt human century visual storytelling depth resilience human spirit weave race gender identity ongoing archive human seek understand remember self reconcile personal struggle universal truth uncover beauty meaning chaos existence objective distribution portrait ancestor fair compensation collaborator aim invite listener shape body root memory legacy cultural storytelling theme andrew intersection identity legacy resilience examine personal cultural history shape carry forward high quality original music professional recording live performance award winning team


In [21]:
# CELLA 6 — controllo: quali bigrammi nuovi sono stati davvero attivati?

feat = np.array(res['feat'])

old_counts = X_text_old.toarray().ravel()
new_counts = X_text_new.toarray().ravel()

added_mask = new_counts > old_counts

added_bigrams_df = pd.DataFrame({
    'bigram': feat[added_mask],
    'count_old': old_counts[added_mask],
    'count_new': new_counts[added_mask],
    'count_added': new_counts[added_mask] - old_counts[added_mask]
}).sort_values('count_added', ascending=False).reset_index(drop=True)

display(added_bigrams_df.head(30))
print('Numero di bigrammi nuovi attivati:', added_bigrams_df.shape[0])

,bigram,count_old,count_new,count_added


Numero di bigrammi nuovi attivati: 0


In [22]:
# CELLA 7 — scegliamo bigrammi esatti sicuramente presenti nel vocabolario

chosen_bigrams = top_success_bigrams.head(8)['bigram'].tolist()

print('Bigrammi scelti:')
for bg in chosen_bigrams:
    print(bg)

# CELLA 8 — costruzione della coda in modo da attivare i bigrammi scelti

bigram_tail = ' '.join(chosen_bigrams)
rewritten_text = original_text + ' ' + bigram_tail

print('Coda aggiunta:')
print(bigram_tail)

# CELLA 9 — nuovo scoring + verifica attivazioni

X_text_new = res['vec'].transform([rewritten_text])
X_num_new  = sp.csr_matrix(res['scaler'].transform(failure_sample[CONTROLS]))
X_new      = hstack([X_text_new, X_num_new])

p_new     = res['model'].predict_proba(X_new)[0, 1]
logit_new = res['model'].decision_function(X_new)[0]

comparison = pd.DataFrame({
    'version': ['original', 'rewritten'],
    'prob_success': [p_old, p_new],
    'logit_score': [logit_old, logit_new]
})

comparison['delta_prob']  = comparison['prob_success'] - comparison.loc[0, 'prob_success']
comparison['delta_logit'] = comparison['logit_score'] - comparison.loc[0, 'logit_score']

display(comparison)

feat = np.array(res['feat'])
old_counts = X_text_old.toarray().ravel()
new_counts = X_text_new.toarray().ravel()

added_mask = new_counts > old_counts

added_bigrams_df = pd.DataFrame({
    'bigram': feat[added_mask],
    'count_old': old_counts[added_mask],
    'count_new': new_counts[added_mask],
    'count_added': new_counts[added_mask] - old_counts[added_mask]
}).sort_values('count_added', ascending=False).reset_index(drop=True)

display(added_bigrams_df.head(20))
print('Numero di bigrammi nuovi attivati:', added_bigrams_df.shape[0])

Bigrammi scelti:
radio station
united state
across country
capable replay
los angeles
rock roll
grammy win
physical cd
Coda aggiunta:
radio station united state across country capable replay los angeles rock roll grammy win physical cd


,version,prob_success,logit_score,delta_prob,delta_logit
0,original,0.335261,-0.684487,0.000000,0.000000
1,rewritten,0.900418,2.201879,0.565157,2.886366


,bigram,count_old,count_new,count_added
0,across country,0,1,1
1,capable replay,0,1,1
2,grammy win,0,1,1
3,los angeles,0,1,1
4,physical cd,0,1,1
5,radio station,0,1,1
6,rock roll,0,1,1
7,united state,0,1,1


Numero di bigrammi nuovi attivati: 8


In [25]:
# ============================================================
# SCRIPT 1 — aggiunta di pochi bigrammi associati al SUCCESSO
# ============================================================

# ------------------------------------------------------------
# 1. Fit intra-categoria bigram e salvataggio oggetti
# ------------------------------------------------------------

gen_models_bi = {}

for cat in CATEGORIES:
    df_cat = df[df['category'] == cat].copy().reset_index(drop=True)
    y_cat  = df_cat['status'].values

    vec    = CountVectorizer(ngram_range=(2, 2), min_df=MIN_DF_BIGRAM)
    X_text = vec.fit_transform(df_cat['text_joined'])
    feat   = vec.get_feature_names_out()

    scaler = StandardScaler()
    X_num  = sp.csr_matrix(scaler.fit_transform(df_cat[CONTROLS]))
    X      = hstack([X_text, X_num])

    model  = make_model()
    model.fit(X, y_cat)

    info = extract_coefs(
        model=model,
        feature_names=feat,
        X_text=X_text,
        min_doc_freq=MIN_DF_BIGRAM,
        col_name='bigram'
    ).reset_index(drop=True)

    gen_models_bi[cat] = {
        'df_cat':  df_cat,
        'y_cat':   y_cat,
        'vec':     vec,
        'X_text':  X_text,
        'feat':    feat,
        'scaler':  scaler,
        'X_num':   X_num,
        'X':       X,
        'model':   model,
        'info':    info
    }

# ------------------------------------------------------------
# 2. Parametri esperimento
# ------------------------------------------------------------

CAT = 'Music'          # cambia categoria
RANDOM_STATE = 42
N_BIGRAMS = 3          # pochi bigrammi pro-successo
MIN_DOC_FREQ_GEN = 20  # filtro opzionale sulla frequenza documentale

# ------------------------------------------------------------
# 3. Estrazione progetto fallito casuale
# ------------------------------------------------------------

res = gen_models_bi[CAT]
df_cat = res['df_cat']

failure_sample = (
    df_cat[df_cat['status'] == 0]
    .sample(1, random_state=RANDOM_STATE)
    .copy()
)

row = failure_sample.iloc[0]
original_text = row['text_joined']

# ------------------------------------------------------------
# 4. Score originale
# ------------------------------------------------------------

X_text_old = res['vec'].transform([original_text])
X_num_old  = sp.csr_matrix(res['scaler'].transform(failure_sample[CONTROLS]))
X_old      = hstack([X_text_old, X_num_old])

p_old     = res['model'].predict_proba(X_old)[0, 1]
logit_old = res['model'].decision_function(X_old)[0]

# ------------------------------------------------------------
# 5. Selezione bigrammi pro-successo
# ------------------------------------------------------------

top_success_bigrams = (
    res['info'][res['info']['doc_freq'] >= MIN_DOC_FREQ_GEN]
    .sort_values(['coef', 'doc_freq'], ascending=[False, False])
    .reset_index(drop=True)
)

chosen_bigrams = top_success_bigrams.head(N_BIGRAMS)['bigram'].tolist()
bigram_tail = ' '.join(chosen_bigrams)

rewritten_text = original_text + ' ' + bigram_tail

# ------------------------------------------------------------
# 6. Score dopo riscrittura
# ------------------------------------------------------------

X_text_new = res['vec'].transform([rewritten_text])
X_num_new  = sp.csr_matrix(res['scaler'].transform(failure_sample[CONTROLS]))
X_new      = hstack([X_text_new, X_num_new])

p_new     = res['model'].predict_proba(X_new)[0, 1]
logit_new = res['model'].decision_function(X_new)[0]

# ------------------------------------------------------------
# 7. Verifica dei bigrammi effettivamente attivati
# ------------------------------------------------------------

feat = np.array(res['feat'])

old_counts = X_text_old.toarray().ravel()
new_counts = X_text_new.toarray().ravel()

added_mask = new_counts > old_counts

added_bigrams_df = pd.DataFrame({
    'bigram': feat[added_mask],
    'count_old': old_counts[added_mask],
    'count_new': new_counts[added_mask],
    'count_added': new_counts[added_mask] - old_counts[added_mask]
}).sort_values('count_added', ascending=False).reset_index(drop=True)

# ------------------------------------------------------------
# 8. Output finale
# ------------------------------------------------------------

comparison = pd.DataFrame({
    'version': ['original', 'rewritten_success'],
    'prob_success': [p_old, p_new],
    'logit_score': [logit_old, logit_new]
})

comparison['delta_prob']  = comparison['prob_success'] - comparison.loc[0, 'prob_success']
comparison['delta_logit'] = comparison['logit_score'] - comparison.loc[0, 'logit_score']

print('Categoria:', CAT)
print('Bigrammi pro-successo aggiunti:')
for bg in chosen_bigrams:
    print('-', bg)

print('\nDescrizione originale:\n')
print(original_text[:1500])

print('\nCoda aggiunta:\n')
print(bigram_tail)

print('\nConfronto probabilità:')
display(comparison)

print('\nBigrammi nuovi effettivamente attivati:')
display(added_bigrams_df.head(20))

print('Numero di bigrammi nuovi attivati:', added_bigrams_df.shape[0])

Categoria: Music
Bigrammi pro-successo aggiunti:
- graphic designer
- radio station
- string quartet

Descrizione originale:

photo henry hung overview portrait ancestor solo singersongwriter andrew across weave grief memory ancestral echo intimate confession layered harmony serve sonic document generation discover study interpret overview andrew hehim capture felt human century visual storytelling depth resilience human spirit weave race gender identity ongoing archive human seek understand remember self reconcile personal struggle universal truth uncover beauty meaning chaos existence objective distribution portrait ancestor fair compensation collaborator aim invite listener shape body root memory legacy cultural storytelling theme andrew intersection identity legacy resilience examine personal cultural history shape carry forward

Coda aggiunta:

graphic designer radio station string quartet

Confronto probabilità:


,version,prob_success,logit_score,delta_prob,delta_logit
0,original,0.335261,-0.684487,0.000000,0.000000
1,rewritten_success,0.991342,4.740547,0.656081,5.425034



Bigrammi nuovi effettivamente attivati:


,bigram,count_old,count_new,count_added
0,graphic designer,0,1,1
1,radio station,0,1,1
2,string quartet,0,1,1


Numero di bigrammi nuovi attivati: 3


In [26]:
# ============================================================
# SCRIPT 2 — aggiunta di pochi bigrammi associati al FALLIMENTO
# ============================================================

# ------------------------------------------------------------
# 1. Fit intra-categoria bigram e salvataggio oggetti
# ------------------------------------------------------------

gen_models_bi = {}

for cat in CATEGORIES:
    df_cat = df[df['category'] == cat].copy().reset_index(drop=True)
    y_cat  = df_cat['status'].values

    vec    = CountVectorizer(ngram_range=(2, 2), min_df=MIN_DF_BIGRAM)
    X_text = vec.fit_transform(df_cat['text_joined'])
    feat   = vec.get_feature_names_out()

    scaler = StandardScaler()
    X_num  = sp.csr_matrix(scaler.fit_transform(df_cat[CONTROLS]))
    X      = hstack([X_text, X_num])

    model  = make_model()
    model.fit(X, y_cat)

    info = extract_coefs(
        model=model,
        feature_names=feat,
        X_text=X_text,
        min_doc_freq=MIN_DF_BIGRAM,
        col_name='bigram'
    ).reset_index(drop=True)

    gen_models_bi[cat] = {
        'df_cat':  df_cat,
        'y_cat':   y_cat,
        'vec':     vec,
        'X_text':  X_text,
        'feat':    feat,
        'scaler':  scaler,
        'X_num':   X_num,
        'X':       X,
        'model':   model,
        'info':    info
    }

# ------------------------------------------------------------
# 2. Parametri esperimento
# ------------------------------------------------------------

CAT = 'Music'          # cambia categoria
RANDOM_STATE = 42
N_BIGRAMS = 3          # pochi bigrammi pro-failure
MIN_DOC_FREQ_GEN = 20  # filtro opzionale sulla frequenza documentale

# ------------------------------------------------------------
# 3. Estrazione progetto fallito casuale
# ------------------------------------------------------------

res = gen_models_bi[CAT]
df_cat = res['df_cat']

failure_sample = (
    df_cat[df_cat['status'] == 0]
    .sample(1, random_state=RANDOM_STATE)
    .copy()
)

row = failure_sample.iloc[0]
original_text = row['text_joined']

# ------------------------------------------------------------
# 4. Score originale
# ------------------------------------------------------------

X_text_old = res['vec'].transform([original_text])
X_num_old  = sp.csr_matrix(res['scaler'].transform(failure_sample[CONTROLS]))
X_old      = hstack([X_text_old, X_num_old])

p_old     = res['model'].predict_proba(X_old)[0, 1]
logit_old = res['model'].decision_function(X_old)[0]

# ------------------------------------------------------------
# 5. Selezione bigrammi pro-failure
# ------------------------------------------------------------

bottom_failure_bigrams = (
    res['info'][res['info']['doc_freq'] >= MIN_DOC_FREQ_GEN]
    .sort_values(['coef', 'doc_freq'], ascending=[True, False])
    .reset_index(drop=True)
)

chosen_bigrams = bottom_failure_bigrams.head(N_BIGRAMS)['bigram'].tolist()
bigram_tail = ' '.join(chosen_bigrams)

rewritten_text = original_text + ' ' + bigram_tail

# ------------------------------------------------------------
# 6. Score dopo riscrittura
# ------------------------------------------------------------

X_text_new = res['vec'].transform([rewritten_text])
X_num_new  = sp.csr_matrix(res['scaler'].transform(failure_sample[CONTROLS]))
X_new      = hstack([X_text_new, X_num_new])

p_new     = res['model'].predict_proba(X_new)[0, 1]
logit_new = res['model'].decision_function(X_new)[0]

# ------------------------------------------------------------
# 7. Verifica dei bigrammi effettivamente attivati
# ------------------------------------------------------------

feat = np.array(res['feat'])

old_counts = X_text_old.toarray().ravel()
new_counts = X_text_new.toarray().ravel()

added_mask = new_counts > old_counts

added_bigrams_df = pd.DataFrame({
    'bigram': feat[added_mask],
    'count_old': old_counts[added_mask],
    'count_new': new_counts[added_mask],
    'count_added': new_counts[added_mask] - old_counts[added_mask]
}).sort_values('count_added', ascending=False).reset_index(drop=True)

# ------------------------------------------------------------
# 8. Output finale
# ------------------------------------------------------------

comparison = pd.DataFrame({
    'version': ['original', 'rewritten_failure'],
    'prob_success': [p_old, p_new],
    'logit_score': [logit_old, logit_new]
})

comparison['delta_prob']  = comparison['prob_success'] - comparison.loc[0, 'prob_success']
comparison['delta_logit'] = comparison['logit_score'] - comparison.loc[0, 'logit_score']

print('Categoria:', CAT)
print('Bigrammi pro-failure aggiunti:')
for bg in chosen_bigrams:
    print('-', bg)

print('\nDescrizione originale:\n')
print(original_text[:1500])

print('\nCoda aggiunta:\n')
print(bigram_tail)

print('\nConfronto probabilità:')
display(comparison)

print('\nBigrammi nuovi effettivamente attivati:')
display(added_bigrams_df.head(20))

print('Numero di bigrammi nuovi attivati:', added_bigrams_df.shape[0])

Categoria: Music
Bigrammi pro-failure aggiunti:
- drum bass
- facebook twitter
- anyone else

Descrizione originale:

photo henry hung overview portrait ancestor solo singersongwriter andrew across weave grief memory ancestral echo intimate confession layered harmony serve sonic document generation discover study interpret overview andrew hehim capture felt human century visual storytelling depth resilience human spirit weave race gender identity ongoing archive human seek understand remember self reconcile personal struggle universal truth uncover beauty meaning chaos existence objective distribution portrait ancestor fair compensation collaborator aim invite listener shape body root memory legacy cultural storytelling theme andrew intersection identity legacy resilience examine personal cultural history shape carry forward

Coda aggiunta:

drum bass facebook twitter anyone else

Confronto probabilità:


,version,prob_success,logit_score,delta_prob,delta_logit
0,original,0.335261,-0.684487,0.000000,0.000000
1,rewritten_failure,0.057556,-2.795715,-0.277705,-2.111229



Bigrammi nuovi effettivamente attivati:


,bigram,count_old,count_new,count_added
0,anyone else,0,1,1
1,drum bass,0,1,1
2,facebook twitter,0,1,1


Numero di bigrammi nuovi attivati: 3


In [27]:
# ============================================================
# AGGIUNTA DI BIGRAMMI ASSOCIATI AL FALLIMENTO
# ============================================================

# ------------------------------------------------------------
# 1. Fit intra-categoria bigram e salvataggio oggetti
# ------------------------------------------------------------

gen_models_bi = {}

for cat in CATEGORIES:
    df_cat = df[df['category'] == cat].copy().reset_index(drop=True)
    y_cat  = df_cat['status'].values

    vec    = CountVectorizer(ngram_range=(2, 2), min_df=MIN_DF_BIGRAM)
    X_text = vec.fit_transform(df_cat['text_joined'])
    feat   = vec.get_feature_names_out()

    scaler = StandardScaler()
    X_num  = sp.csr_matrix(scaler.fit_transform(df_cat[CONTROLS]))
    X      = hstack([X_text, X_num])

    model  = make_model()
    model.fit(X, y_cat)

    info = extract_coefs(
        model=model,
        feature_names=feat,
        X_text=X_text,
        min_doc_freq=MIN_DF_BIGRAM,
        col_name='bigram'
    ).reset_index(drop=True)

    gen_models_bi[cat] = {
        'df_cat':  df_cat,
        'y_cat':   y_cat,
        'vec':     vec,
        'X_text':  X_text,
        'feat':    feat,
        'scaler':  scaler,
        'X_num':   X_num,
        'X':       X,
        'model':   model,
        'info':    info
    }

# ------------------------------------------------------------
# 2. Parametri esperimento
# ------------------------------------------------------------

CAT = 'Music'           # cambia categoria
RANDOM_STATE = 42
N_BIGRAMS = 8           # qui puoi aumentare liberamente
MIN_DOC_FREQ_GEN = 20   # filtro opzionale sulla frequenza documentale

# ------------------------------------------------------------
# 3. Estrazione progetto fallito casuale
# ------------------------------------------------------------

res = gen_models_bi[CAT]
df_cat = res['df_cat']

failure_sample = (
    df_cat[df_cat['status'] == 0]
    .sample(1, random_state=RANDOM_STATE)
    .copy()
)

row = failure_sample.iloc[0]
original_text = row['text_joined']

# ------------------------------------------------------------
# 4. Score originale
# ------------------------------------------------------------

X_text_old = res['vec'].transform([original_text])
X_num_old  = sp.csr_matrix(res['scaler'].transform(failure_sample[CONTROLS]))
X_old      = hstack([X_text_old, X_num_old])

p_old     = res['model'].predict_proba(X_old)[0, 1]
logit_old = res['model'].decision_function(X_old)[0]

# ------------------------------------------------------------
# 5. Selezione bigrammi associati al fallimento
# ------------------------------------------------------------

failure_bigrams = (
    res['info'][res['info']['doc_freq'] >= MIN_DOC_FREQ_GEN]
    .sort_values(['coef', 'doc_freq'], ascending=[True, False])
    .reset_index(drop=True)
)

chosen_bigrams = failure_bigrams.head(N_BIGRAMS)['bigram'].tolist()
bigram_tail = ' '.join(chosen_bigrams)

rewritten_text = original_text + ' ' + bigram_tail

# ------------------------------------------------------------
# 6. Score dopo aggiunta dei bigrammi failure
# ------------------------------------------------------------

X_text_new = res['vec'].transform([rewritten_text])
X_num_new  = sp.csr_matrix(res['scaler'].transform(failure_sample[CONTROLS]))
X_new      = hstack([X_text_new, X_num_new])

p_new     = res['model'].predict_proba(X_new)[0, 1]
logit_new = res['model'].decision_function(X_new)[0]

# ------------------------------------------------------------
# 7. Verifica dei bigrammi effettivamente attivati
# ------------------------------------------------------------

feat = np.array(res['feat'])

old_counts = X_text_old.toarray().ravel()
new_counts = X_text_new.toarray().ravel()

added_mask = new_counts > old_counts

added_bigrams_df = pd.DataFrame({
    'bigram': feat[added_mask],
    'count_old': old_counts[added_mask],
    'count_new': new_counts[added_mask],
    'count_added': new_counts[added_mask] - old_counts[added_mask]
}).sort_values('count_added', ascending=False).reset_index(drop=True)

# ------------------------------------------------------------
# 8. Contributo approssimato al logit dei bigrammi aggiunti
# ------------------------------------------------------------

coef_map = dict(zip(res['info']['bigram'], res['info']['coef']))

if not added_bigrams_df.empty:
    added_bigrams_df['coef'] = added_bigrams_df['bigram'].map(coef_map)
    added_bigrams_df['approx_logit_contribution'] = (
        added_bigrams_df['count_added'] * added_bigrams_df['coef']
    )
    added_bigrams_df = added_bigrams_df.sort_values(
        'approx_logit_contribution',
        ascending=True
    ).reset_index(drop=True)

# ------------------------------------------------------------
# 9. Output finale
# ------------------------------------------------------------

comparison = pd.DataFrame({
    'version': ['original', 'rewritten_failure'],
    'prob_success': [p_old, p_new],
    'logit_score': [logit_old, logit_new]
})

comparison['delta_prob']  = comparison['prob_success'] - comparison.loc[0, 'prob_success']
comparison['delta_logit'] = comparison['logit_score'] - comparison.loc[0, 'logit_score']

print('Categoria:', CAT)
print('\nBigrammi associati al fallimento aggiunti:')
for bg in chosen_bigrams:
    print('-', bg)

print('\nDescrizione originale:\n')
print(original_text[:1500])

print('\nCoda aggiunta:\n')
print(bigram_tail)

print('\nConfronto probabilità:')
display(comparison)

print('\nBigrammi nuovi effettivamente attivati:')
display(added_bigrams_df.head(30))

print('Numero di bigrammi nuovi attivati:', added_bigrams_df.shape[0])

Categoria: Music

Bigrammi associati al fallimento aggiunti:
- drum bass
- facebook twitter
- anyone else
- countless hour
- singer songwriter
- marketing promotion
- thus far
- grammy winner

Descrizione originale:

photo henry hung overview portrait ancestor solo singersongwriter andrew across weave grief memory ancestral echo intimate confession layered harmony serve sonic document generation discover study interpret overview andrew hehim capture felt human century visual storytelling depth resilience human spirit weave race gender identity ongoing archive human seek understand remember self reconcile personal struggle universal truth uncover beauty meaning chaos existence objective distribution portrait ancestor fair compensation collaborator aim invite listener shape body root memory legacy cultural storytelling theme andrew intersection identity legacy resilience examine personal cultural history shape carry forward

Coda aggiunta:

drum bass facebook twitter anyone else countles

,version,prob_success,logit_score,delta_prob,delta_logit
0,original,0.335261,-0.684487,0.000000,0.000000
1,rewritten_failure,0.011447,-4.458548,-0.323814,-3.774061



Bigrammi nuovi effettivamente attivati:


,bigram,count_old,count_new,count_added,coef,approx_logit_contribution
0,drum bass,0,1,1,-0.8314,-0.8314
1,facebook twitter,0,1,1,-0.7418,-0.7418
2,anyone else,0,1,1,-0.5381,-0.5381
3,countless hour,0,1,1,-0.4723,-0.4723
4,singer songwriter,0,1,1,-0.4044,-0.4044
5,marketing promotion,0,1,1,-0.3334,-0.3334
6,thus far,0,1,1,-0.2586,-0.2586
7,grammy winner,0,1,1,-0.1941,-0.1941


Numero di bigrammi nuovi attivati: 8


### The generative experiment provides a clear validation of the structure learned by the model. By taking failed projects and systematically modifying their descriptions through the addition of selected bigrams, the predicted probability of success changes in a precise and predictable way.
### The key result is the strong additivity of the model. Each bigram contributes linearly to the latent score, and the total change in the log-odds is approximately equal to the sum of the coefficients of the added terms. This is directly observed in the decomposition of the logit variation, where each inserted bigram shifts the score by its estimated coefficient. When multiple bigrams are added, their effects accumulate without interaction, producing large movements in the predicted probability.
### Closely related is the property of monotonicity. Adding bigrams associated with success systematically increases the predicted probability, while adding bigrams associated with failure decreases it. Moreover, the magnitude of the change grows with the number and strength of the inserted terms, generating a clear dose–response relationship. This confirms that the model encodes a consistent directional signal in the language features.
### At the same time, the experiment highlights an important limitation. The model responds purely to the presence of specific lexical patterns, without evaluating the coherence or semantic meaning of the text. As a consequence, artificially appending a small number of high-weight bigrams can dramatically alter the prediction, even though the underlying project content remains unchanged. This indicates that the model captures a statistical association between language and outcomes, rather than a causal effect of wording on success.
### Overall, the results show that linguistic patterns carry substantial predictive information within categories, and that this information is encoded in a linear and highly interpretable way. However, they also reveal the fragility of bag-of-words representations to superficial manipulations, motivating the use of more context-aware models in subsequent analysis.